In [1]:
# ==========================================
# 1. Standard Library Imports
# ==========================================
import os
import sys
import csv
import time
import pickle
import random
import warnings
import itertools
import textwrap
import re
from collections import Counter
import requests
from io import StringIO

# ==========================================
# 2. Third-Party Scientific Imports
# ==========================================
import numpy as np
import pandas as pd
import xarray as xr
import scipy.stats as stats
import scipy.linalg as linalg
import scipy.optimize as optimize
import scipy.fft as fft
import scipy.signal as signal
from scipy.interpolate import interp1d
from scipy.stats import pearsonr, skewnorm

# Specialized Science Toolkits
import pywt
import pycwt as wavelet
import emd
from eofs.standard import Eof
import shap
import lime
from lime.lime_tabular import LimeTabularExplainer

# Machine Learning (Scikit-Learn)
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, PolynomialFeatures, normalize
from sklearn.model_selection import (
    train_test_split, GridSearchCV, StratifiedShuffleSplit, 
    cross_val_score, KFold, LeaveOneOut, StratifiedKFold
)
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, VotingClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score, 
    roc_curve, accuracy_score, classification_report
)
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn import tree
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.cluster import hierarchy

# ==========================================
# 3. Visualization & Plotting
# ==========================================
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as patches
import matplotlib.patheffects as PathEffects
import seaborn as sns
import imgkit
from matplotlib.ticker import ScalarFormatter
from matplotlib.colors import BoundaryNorm

# Geospatial Plotting
import cartopy
import cartopy.crs as ccrs
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.util import add_cyclic_point

# ==========================================
# 4. Configuration & Local Modules
# ==========================================
warnings.filterwarnings('ignore')
sns.set_theme() # Optional: sets a cleaner default style for plots

# Define Paths
HURDAT_CODE_PATH = '/Users/jackskari/Desktop/NCSU_Research/Modularized_Code/'
DATA_FILE_PATH = "/Users/jackskari/Desktop/NCSU_Research/New_Data_2025/hurr2025.csv"

# Dynamic import of local module
if HURDAT_CODE_PATH not in sys.path:
    sys.path.append(HURDAT_CODE_PATH)

try:
    import ReadHurdatData
except ImportError:
    print(f"Error: Could not find ReadHurdatData at {HURDAT_CODE_PATH}")

# ==========================================
# 5. Data Loading & Initial Processing
# ==========================================
df = pd.read_csv(DATA_FILE_PATH)

# Load Near-Land Data
(nearmod_df, nearseasonal_ace, nearseasonal_num, nearseasonal_dur, 
 nearseasonal_apsm, nearseasonal_ap6h, nearseasonal_avdur, nearmonthly_ace, 
 nearseasonal_avmaxwind, nearseasonal_hurr, nearseasonal_mh, nearseasonal_hu, 
 nearseasonal_mjhu, yearz) = ReadHurdatData.hurdatclean(df, 1, 12, landmask=1, nearland=2)

# Load Standard Seasonal Data
(mod_df, seasonal_ace, seasonal_num, seasonal_dur, 
 seasonal_apsm, seasonal_ap6h, seasonal_avdur, monthly_ace, 
 seasonal_avmaxwind, seasonal_hurr, seasonal_mh, seasonal_hu, 
 seasonal_mjhu, yearz) = ReadHurdatData.hurdatclean(df, 1, 12)

# Slicing Indices (100+)
yearzz = np.asarray(yearz[100:])
ace_prime = seasonal_ace[100:]
num_prime = seasonal_num[100:]

nearace_prime = nearseasonal_ace[100:]
nearnum_prime = nearseasonal_num[100:]

# Calling in the Initial Data

In [25]:
%%time
# --- 1. Global Configuration ---
STR_YR = 1950
END_YR = 2025
MONTH_COLS = ["JAN","FEB","MAR","APR","MAY","JUN","JUL","AUG","SEP","OCT","NOV","DEC"]
BASE_PATH = "/Users/jackskari/Desktop/NCSU_Research/New_Data_2025/"

# --- 2. Unified Helper Functions ---

def clean_climate_df(df, year_col='year'):
    """Standardizes column names and drops extra columns."""
    if df.shape[1] > 13:
        df = df.iloc[:, :13]
    df.columns = ["year"] + MONTH_COLS
    return df.apply(pd.to_numeric, errors='coerce').reset_index(drop=True)

def fetch_noaa_data(url, n_lines=79, drop_meta=3):
    """Refactored readhttp: Handles standard NOAA .data formatted files."""
    response = requests.get(url)
    lines = response.text.splitlines()[:n_lines]
    content = "\n".join(lines[drop_meta:])
    df = pd.read_csv(StringIO(content), sep='\s+', header=None)
    return clean_climate_df(df)

def load_local_mdr(filename,skiprow=2):
    """Standard loader for your local MDR text files."""
    df = pd.read_csv(f"{BASE_PATH}{filename}", sep='\s+', skiprows=skiprow, header=None)
    return clean_climate_df(df).iloc[:-1] # Remove footer/last row

# --- 3. Remote Data Ingestion ---

# Standard NOAA ESRL formats
nao    = fetch_noaa_data("http://www.esrl.noaa.gov/psd/data/correlation/nao.data")
pna    = fetch_noaa_data("http://www.esrl.noaa.gov/psd/data/correlation/pna.data")
qbo    = fetch_noaa_data("http://www.esrl.noaa.gov/psd/data/correlation/qbo.data")
tna    = fetch_noaa_data("http://www.esrl.noaa.gov/psd/data/correlation/tna.data")
tsa    = fetch_noaa_data("http://www.esrl.noaa.gov/psd/data/correlation/tsa.data")
whwp   = fetch_noaa_data("http://www.esrl.noaa.gov/psd/data/correlation/whwp.data")
nino12 = fetch_noaa_data("http://www.esrl.noaa.gov/psd/data/correlation/nina1.data")
nino3 = fetch_noaa_data("http://www.esrl.noaa.gov/psd/data/correlation/nina3.data")
nino34 = fetch_noaa_data("http://www.esrl.noaa.gov/psd/data/correlation/nina34.data")
nino4 = fetch_noaa_data("http://www.esrl.noaa.gov/psd/data/correlation/nina4.data")
censo = fetch_noaa_data("http://www.esrl.noaa.gov/psd/data/correlation/censo.data")
tni    = fetch_noaa_data("http://www.esrl.noaa.gov/psd/data/correlation/tni.data")
# Fix NOV and DEC
last_row = len(tni)
avg_rest = tni.loc[:last_row - 2, "NOV"].mean(skipna=True)
tni.loc[last_row - 1, "NOV"] = round(avg_rest, 3)
avg_rest = tni.loc[:last_row - 2, "DEC"].mean(skipna=True)
tni.loc[last_row - 1, "DEC"] = round(avg_rest, 3)
soi = fetch_noaa_data("http://www.esrl.noaa.gov/psd/data/correlation/soi.data")
# Fix with Older SOI Data
soi.loc[0] = np.array([1950.0, 0.5,   2.1,   1.9,   1.2,   0.6,   2.0,   2.0,   1.1,   0.7,   1.6,   1.0 ,  2.7])
wp = fetch_noaa_data("http://www.esrl.noaa.gov/psd/data/correlation/wp.data")
epo = fetch_noaa_data("http://www.esrl.noaa.gov/psd/data/correlation/epo.data")
epo["DEC"]=round((epo["JAN"]+epo["FEB"]+epo["MAR"]+epo["APR"]+epo["MAY"]+epo["JUN"]+epo["JUL"]+epo["AUG"]+epo["SEP"]+epo["OCT"]+epo["NOV"])/11,2)

# Special Handling for AMM/AMO (Reshaping formats)
def fetch_reshaped(url, value_col, skiprow):
    df_raw = pd.read_csv(url, sep='\s+', skiprows=skiprow)
    df_filt = df_raw[(df_raw['Year'] >= STR_YR) & (df_raw['Year'] <= END_YR)]
    values = df_filt[value_col].values.reshape(-1, 12)
    new_df = pd.DataFrame(values, columns=MONTH_COLS)
    new_df.insert(0, 'year', np.arange(STR_YR, END_YR + 1))
    return new_df


amm = fetch_reshaped("https://www.aos.wisc.edu/dvimont/MModes/RealTime/AMM.txt", 'SST', 0)

amo = fetch_reshaped("https://www1.ncdc.noaa.gov/pub/data/cmb/ersst/v5/index/ersst.v5.amo.dat", 'SSTA', 1)


# --- 4. Local Data Ingestion ---

# These were taken from the NOAA PSL

mdrsst  = load_local_mdr("MDRSST.txt") # 10-20 N, 20-80 W
mdrslp  = load_local_mdr("MDRSLP.txt") # 10-20 N, 20-80 W
mdrolr  = load_local_mdr("MDROLR.txt") # 10-20 N, 20-80 W
mdru200 = load_local_mdr("MDRU200.txt") # 10-20 N, 20-80 W
mdru850 = load_local_mdr("MDRU850.txt") # 10-20 N, 20-80 W
mdrv200 = load_local_mdr("MDRV200.txt") # 10-20 N, 20-80 W
mdrv850 = load_local_mdr("MDRV850.txt") # 10-20 N, 20-80 W

mdrt200  = load_local_mdr("MDRT200.txt") # 10-20 N, 20-80 W
gomslp  = load_local_mdr("GOMSLP.txt") # 20-30 N, 80-100 W
gomolr  = load_local_mdr("GOMOLR.txt") # 20-30 N, 80-100 W
gomu200 = load_local_mdr("GOMU200.txt") # 20-30 N, 80-100 W
gomu850 = load_local_mdr("GOMU850.txt") # 20-30 N, 80-100 W
gomv200 = load_local_mdr("GOMV200.txt") # 20-30 N, 80-100 W
gomv850 = load_local_mdr("GOMV850.txt") # 20-30 N, 80-100 W

roni = load_local_mdr("roni.data.txt",1)
rhsal = load_local_mdr("RHSAL.txt") # 10-30 N, 15-65 W
# --- 5. Calculated Indices (Vectorized) ---

def calc_wind_shear(u_high, u_low, v_high, v_low):
    u_diff = u_high.iloc[:, 1:].values - u_low.iloc[:, 1:].values
    v_diff = v_high.iloc[:, 1:].values - v_low.iloc[:, 1:].values
    shear_vals = np.sqrt(u_diff**2 + v_diff**2)
    df = pd.DataFrame(shear_vals, columns=MONTH_COLS)
    df.insert(0, 'year', u_high['year'].values)
    return df

mdrvws = calc_wind_shear(mdru200, mdru850, mdrv200, mdrv850)
gomvws = calc_wind_shear(gomu200, gomu850, gomv200, gomv850)
dm = tna.copy()
dm.iloc[:, 1:] = tna.iloc[:, 1:].values - tsa.iloc[:, 1:].values

# --- 6. NetCDF / Sahel Precipitation ---

def process_sahel_precip(nc_path):
    ds = xr.open_dataset(nc_path)
    # Fix longitude -180 to 180
    ds = ds.assign_coords(lon=(((ds.lon + 180) % 360) - 180)).sortby('lon')
    
    # Slice time and region
    subset = ds.sel(time=slice(f"{STR_YR}-01-01", f"{END_YR}-12-01"),
                    lat=slice(20, 5), lon=slice(-20, 20))
    
    avg_ts = subset.precip.mean(("lat", "lon")).fillna(0).values
    reshaped = avg_ts.reshape(-1, 12)
    return pd.DataFrame(reshaped, columns=MONTH_COLS)

sahel = process_sahel_precip(f"{BASE_PATH}precip.comb.v2020to2019-v2020monitorafter.total.nc")

mei = pd.read_csv("/Users/jackskari/Desktop/NCSU_Research/New_Data_2025/meiv2.txt", sep='\s+', header=None)
mei.columns = ["year", "JAN","FEB","MAR","APR",
               "MAY","JUN","JUL","AUG",
               "SEP","OCT","NOV","DEC"]
mei = mei.iloc[:-1]

pdo_file = "/Users/jackskari/Desktop/NCSU_Research/New_Data_2025/ersst.pdo.txt"
with open(pdo_file, 'r') as f:
    total_lines = len(f.readlines())

pdo = pd.read_csv(pdo_file, skiprows=3, nrows=total_lines - 3, sep='\s+', header=None)
pdo.columns = ["year", "JAN","FEB","MAR","APR",
               "MAY","JUN","JUL","AUG",
               "SEP","OCT","NOV","DEC"]
pdo = pdo.iloc[:-1]

sfi = pd.read_csv("/Users/jackskari/Desktop/NCSU_Research/New_Data_2025/solar.txt", sep='\s+',skiprows=2, skipfooter=9)
sfi.columns = ["year", "JAN","FEB","MAR","APR",
               "MAY","JUN","JUL","AUG",
               "SEP","OCT","NOV","DEC"]

ao_url = "http://www.cpc.ncep.noaa.gov/products/precip/CWlink/daily_ao_index/monthly.ao.index.b50.current.ascii.table"

# Get raw text
response = requests.get(ao_url)
lines = response.text.splitlines()

# Remove header lines 0, and footer lines 76,77
data_lines = lines[1:77]

# Split each line manually by whitespace and filter empty strings
data_split = [list(filter(None, line.split())) for line in data_lines]

# Convert to DataFrame
ao = pd.DataFrame(data_split, columns=["year", "JAN","FEB","MAR","APR",
                                      "MAY","JUN","JUL","AUG",
                                      "SEP","OCT","NOV","DEC"])

# Convert all to numeric
ao = ao.apply(pd.to_numeric)

oni_url = "http://www.esrl.noaa.gov/psd/data/correlation/oni.data"
oni_lines = requests.get(oni_url).text.splitlines()
oni_data = oni_lines[1:77]  # skip first 2 lines, read next 76

# Parse, reshape, drop last column
oni_array = np.genfromtxt(StringIO("\n".join(oni_data)), dtype=float)
oni = pd.DataFrame(oni_array, columns=["year", "JAN","FEB","MAR","APR",
                                         "MAY","JUN","JUL","AUG",
                                         "SEP","OCT","NOV","DEC"])

def read_gistemp_fixed_width(filename):
    # Column widths: 6 for year, then 12 x 5 = 60 for months, 6 x 5 = 30 extra columns, last 6
    # We will only keep first 13 columns (year + 12 months)
    colspecs = [(0,6)] + [(6 + 5*i, 11 + 5*i) for i in range(18)]  # read 19 columns total
    
    df_raw = pd.read_fwf(filename, colspecs=colspecs, skiprows=83)

    # Drop rows at indices 11,12,33,34,55,56,77,78 and 83 to 99 (0-based)
    drop_rows = list(range(11,12)) + list(range(32,33)) + list(range(53,54)) + list(range(74,75)) + list(range(80,100))
    df = df_raw.drop(drop_rows, errors='ignore')

    # Drop columns after 13th (keep year + 12 months)
    df = df.iloc[:, :13]

    # Convert all columns to numeric (they may be objects)
    df = df.apply(pd.to_numeric, errors='coerce')

    # Rename columns
    df.columns = ["year", "JAN","FEB","MAR","APR",
                  "MAY","JUN","JUL","AUG",
                  "SEP","OCT","NOV","DEC"]

    return df.reset_index(drop=True)

ggst = read_gistemp_fixed_width("/Users/jackskari/Desktop/NCSU_Research/New_Data_2025/ggst.txt")
ngst = read_gistemp_fixed_width("/Users/jackskari/Desktop/NCSU_Research/New_Data_2025/ngst.txt")
sgst = read_gistemp_fixed_width("/Users/jackskari/Desktop/NCSU_Research/New_Data_2025/sgst.txt")

def fetch_reshaped_nino(url, value_col, skiprow):
    df_raw = pd.read_csv(url, sep='\s+', skiprows=skiprow)
    df_filt = df_raw[(df_raw['YR'] >= STR_YR) & (df_raw['YR'] <= END_YR)]
    values = df_filt.iloc[:, value_col].values.reshape(-1, 12)
    new_df = pd.DataFrame(values, columns=MONTH_COLS)
    new_df.insert(0, 'year', np.arange(STR_YR, END_YR + 1))
    return new_df

nino12anom = fetch_reshaped_nino("https://www.cpc.ncep.noaa.gov/data/indices/ersst5.nino.mth.91-20.ascii", 3, 0)
nino3anom = fetch_reshaped_nino("https://www.cpc.ncep.noaa.gov/data/indices/ersst5.nino.mth.91-20.ascii", 5, 0)
nino34anom = fetch_reshaped_nino("https://www.cpc.ncep.noaa.gov/data/indices/ersst5.nino.mth.91-20.ascii", 7, 0)
nino4anom = fetch_reshaped_nino("https://www.cpc.ncep.noaa.gov/data/indices/ersst5.nino.mth.91-20.ascii", 9, 0)
realnino34anom = fetch_reshaped_nino("https://www.cpc.ncep.noaa.gov/products/analysis_monitoring/ensostuff/detrend.nino34.ascii.txt",4,0)

CPU times: user 23.5 s, sys: 784 ms, total: 24.3 s
Wall time: 34.6 s


# Remove the Influence of ENSO from Upper Level Heating and Atlantic Winds (Optional)

In [26]:
# If I want to eliminate ENSO from the wind vectors and upper level heating
# Taken from NOAA PSL NCEP/NCAR Reanalysis
file=xr.open_dataset(f"{BASE_PATH}uwnd.mon.mean.nc")
lon_name='lon'
file['_longitude_adjusted'] = xr.where(file[lon_name] > 180, file[lon_name] - 360, file[lon_name])
file = file.swap_dims({lon_name: '_longitude_adjusted'})
file = file.sel(**{'_longitude_adjusted': sorted(file._longitude_adjusted)})
index1=file.indexes['time'].get_loc('1950-01-01T00:00:00.000000000')
index2=file.indexes['time'].get_loc('2025-12-01T00:00:00.000000000')
file=file.isel(time=slice(index1,index2+1))
uwindfile = file.drop_vars(lon_name)
uwindfile = uwindfile.rename({'_longitude_adjusted': lon_name})

# Taken from NOAA PSL NCEP/NCAR Reanalysis
file=xr.open_dataset(f"{BASE_PATH}vwnd.mon.mean.nc")
lon_name='lon'
file['_longitude_adjusted'] = xr.where(file[lon_name] > 180, file[lon_name] - 360, file[lon_name])
file = file.swap_dims({lon_name: '_longitude_adjusted'})
file = file.sel(**{'_longitude_adjusted': sorted(file._longitude_adjusted)})
index1=file.indexes['time'].get_loc('1950-01-01T00:00:00.000000000')
index2=file.indexes['time'].get_loc('2025-12-01T00:00:00.000000000')
file=file.isel(time=slice(index1,index2+1))
vwindfile = file.drop_vars(lon_name)
vwindfile = vwindfile.rename({'_longitude_adjusted': lon_name})

# Taken from NOAA PSL NCEP/NCAR Reanalysis
file=xr.open_dataset(f"{BASE_PATH}air.mon.mean.nc")
lon_name='lon'
file['_longitude_adjusted'] = xr.where(file[lon_name] > 180, file[lon_name] - 360, file[lon_name])
file = file.swap_dims({lon_name: '_longitude_adjusted'})
file = file.sel(**{'_longitude_adjusted': sorted(file._longitude_adjusted)})
index1=file.indexes['time'].get_loc('1950-01-01T00:00:00.000000000')
index2=file.indexes['time'].get_loc('2025-12-01T00:00:00.000000000')
file=file.isel(time=slice(index1,index2+1))
airfile = file.drop_vars(lon_name)
airfile = airfile.rename({'_longitude_adjusted': lon_name})

allmonth=['DEC','JAN','FEB','MAR','APR','MAY','JUN','JUL','AUG','SEP','OCT','NOV']
earlyyear=['JAN','FEB','MAR','APR','MAY']

In [27]:
%%time
def nino_filter_wind(wind_data, nino_index, lats,pcnum):
    """
    Vectorized ENSO signal removal using EOF decomposition.
    """
    # 1. Prepare Weights (Cosine of Latitude)
    coslat = np.cos(np.deg2rad(lats))
    wgts = np.sqrt(coslat)[..., np.newaxis]
    
    # 2. EOF Solver
    # Normalize the data (standard anomalies)

    wind_std = wind_data.std(axis=0)
    wind_mean = wind_data.mean(axis=0)
    wind_norm = (wind_data - wind_mean) / wind_std
    solver = Eof(wind_norm, weights=wgts)

    # Get all PCs and EOFs
    pcs = solver.pcs(npcs=pcnum, pcscaling=1)
    eofs = solver.eofs(neofs=pcnum)
    
    # 3. Correlation Masking (Vectorized Pearson)
    # We identify PCs that correlate significantly with RONI
    keep_indices = []
    for i in range(pcs.shape[1]):
        corr, p_val = pearsonr(nino_index, pcs[:, i])
        if p_val >= 0.05: # Keep only those NOT significantly correlated
            keep_indices.append(i)
            
    # 4. Reconstruction
    selected_pcs = pcs[:, keep_indices]
    selected_eofs = eofs[keep_indices]
    
    # Reconstruct back to physical space: (Time x PCs) dot (PCs x Space)
    
    recon = np.tensordot(selected_pcs, selected_eofs, axes=((1), (0)))
    # Re-apply mean and standard deviation, then take spatial mean
    final_spat_mean = (recon * wind_std + wind_mean).mean(axis=(1, 2))
    return final_spat_mean

def process_wind_component(ds, var_name, roni_df, regions, levels):
    """Processes all months and regions for a single wind component (U or V)."""
    results = {region_name: [] for region_name in regions}
    
    for month_idx, month_name in enumerate(allmonth, 1):
        # Slice only the relevant month
        month_ds = ds.sel(time=ds.time.dt.month == month_idx, level = levels)
        nino_month = roni_df[month_name].values
        pcnumber=(ds["time.year"].max().item()+1)-ds["time.year"].min().item()
        
        for name, area in regions.items():
            subset = month_ds.sel(lat=area['lat'], lon=area['lon'])

            filtered_ts = nino_filter_wind(subset[var_name].values, nino_month, subset.lat.values,pcnumber)
            results[name].append(filtered_ts)
            
    # Convert lists to DataFrames
    final_dfs = {}
    for name, area in regions.items():
        monthly_data = []
        subset = ds.sel(lat=area['lat'], lon=area['lon'])
        
        for month_idx, month_name in enumerate(allmonth, 1):
            # 1. Filter month and get RONI index for that month
            month_ds = subset.sel(time=subset.time.dt.month == month_idx, level = levels)
            nino_month = roni_df[month_name].values
            
            # 2. Run the EOF filter
            # This returns a 1D array of length 76 (one value per year)
            filtered_ts = nino_filter_wind(month_ds[var_name].values, nino_month, subset.lat.values,pcnumber)
            monthly_data.append(filtered_ts)
            
        # 3. Transpose from (12, 76) to (76, 12) to match Year/Month structure
        combined_matrix = np.array(monthly_data).T 
        
        df = pd.DataFrame(combined_matrix, columns=allmonth)
        df.insert(0, 'year', np.unique(ds.time.dt.year))
        final_dfs[name] = df
            
    return final_dfs
# --- Execution ---

# Define Regions for a single loop
areas = {
    'mdr': {'lat': slice(20, 10), 'lon': slice(-80, -20)},
    'gom': {'lat': slice(30, 20), 'lon': slice(-100, -80)}
}

# Process U and V
# (Using the uwindfile and vwindfile created in your previous block)
u_results200 = process_wind_component(uwindfile, 'uwnd', roni, areas, 200)
v_results200 = process_wind_component(vwindfile, 'vwnd', roni, areas, 200)
u_results850 = process_wind_component(uwindfile, 'uwnd', roni, areas, 850)
v_results850 = process_wind_component(vwindfile, 'vwnd', roni, areas, 850)
air_results200 = process_wind_component(airfile, 'air', roni, areas, 200)

# --- Final Wind Shear Calculation ---
# Now calculate VWS using the ENSO-free components
def calc_vws_clean(u200, u850, v200, v850):
    u_diff = u200.iloc[:, 1:].values - u850.iloc[:, 1:].values
    v_diff = v200.iloc[:, 1:].values - v850.iloc[:, 1:].values
    shear = np.sqrt(u_diff**2 + v_diff**2)
    return pd.DataFrame(shear, columns=allmonth)

mdr_vws_clean = calc_vws_clean(u_results200['mdr'], u_results850['mdr'], v_results200['mdr'], v_results850['mdr'])
gom_vws_clean = calc_vws_clean(u_results200['gom'], u_results850['gom'], v_results200['gom'], v_results850['gom'])

CPU times: user 13.3 s, sys: 45.5 s, total: 58.8 s
Wall time: 8.14 s


# Dump Initial Datasets

In [28]:
# Save all of the variables into files
all_vars = {
    'amm': amm, 'amo': amo, 'ao': ao, 'censo': censo, 'dm': dm, 'epo': epo, 'ggst': ggst, 'ngst': ngst, 'sgst': sgst,
    'mdrolr': mdrolr, 'mdrslp': mdrslp, 'mdrsst': mdrsst, 'nao': nao, 'pdo': pdo, 'pna': pna,  
    'qbo': qbo, 'sfi': sfi, 'soi': soi, 'tni': tni, 'tna': tna, 'tsa': tsa,
    'whwp': whwp, 'wpi': wp, 'nino12': nino12, 'nino3': nino3, 'nino34': nino34, 'nino4': nino4, 'mei': mei, 'roni': roni,
    'rhsal': rhsal, 'gomslp': gomslp, 'gomolr': gomolr
}
# , 'nino12anom': nino12anom, 'nino3anom': nino3anom, 'nino34anom': nino34anom, 'nino4anom': nino4anom
# Include ENSO Forecast
allave1 = {k: v.iloc[:, 1:] for k, v in all_vars.items()}
# We want both at our disposal so save both under different file names
NEW_PATH = "/Users/jackskari/Desktop/NCSU_Research/ML_Data_Classes/"
ensoremover='OFF'
if ensoremover=='ON':
    allave1['mdrt200']=air_results200['mdr']
    allave1['sahel']=sahel
    allave1['mdru200']=u_results200['mdr']
    allave1['mdru850']=u_results850['mdr']
    allave1['mdrv200']=v_results200['mdr']
    allave1['mdrv850']=v_results850['mdr']
    allave1['mdrvws']=mdr_vws_clean
    
    allave1['gomu200']=u_results200['gom']
    allave1['gomu850']=u_results850['gom']
    allave1['gomv200']=v_results200['gom']
    allave1['gomv850']=v_results850['gom']
    allave1['gomvws']=gom_vws_clean
    with open(f"{NEW_PATH}ensoremoveddata1950-2025.pkl", "wb") as f:
        pickle.dump(allave1, f)

elif ensoremover=='OFF':
    allave1['mdrt200']=mdrt200.iloc[:, 1:]
    allave1['sahel']=sahel
    allave1['mdru200']=mdru200
    allave1['mdru850']=mdru850
    allave1['mdrv200']=mdrv200
    allave1['mdrv850']=mdrv850
    allave1['mdrvws']=mdrvws

    allave1['gomu200']=gomu200
    allave1['gomu850']=gomu850
    allave1['gomv200']=gomv200
    allave1['gomv850']=gomv850
    allave1['gomvws']=gomvws
    with open(f"{NEW_PATH}truedata1950-2025.pkl", "wb") as f:
        pickle.dump(allave1, f)

# Reload Datasets and Rearrange For Pre Seasonal Values

In [29]:
neomonth=['JUN','JUL','AUG','SEP','OCT','NOV','DEC']
feature_names = ['amm', 'amo', 'ao', 'censo', 'dm', 'epo', 'ggst', 'ngst', 'sgst',
    'mdrolr', 'mdrslp', 'mdrsst', 'nao', 'pdo', 'pna', 'qbo', 'sfi', 'soi', 'tni', 
    'tna', 'tsa', 'whwp', 'wpi', 'nino12', 'nino3', 'nino34', 'nino4', 'mei', 'roni', 
    'sahel', 'mdru200', 'mdru850', 'mdrv200', 'mdrv850', 'mdrvws', 'gomu200', 'gomu850',
    'gomv200', 'gomv850', 'gomvws', 'mdrt200', 'gomolr', 'gomslp', 'rhsal'
]
ensoremover='OFF' # If you want to use ENSO removed winds or not
if ensoremover=='ON':
    with open(f"{NEW_PATH}ensoremoveddata1950-2025.pkl", 'rb') as file:
        datapolish = pickle.load(file)    
elif ensoremover=='OFF':
    with open(f"{NEW_PATH}truedata1950-2025.pkl", 'rb') as file:
        datapolish = pickle.load(file)   

# Rearrange data to allow for previous year month values
for i in feature_names:
    for j in neomonth[::-1]:
        datapolish[i][j]=datapolish[i][j].shift(1)
        column_to_move = datapolish[i].pop(j)
        datapolish[i].insert(0, j, column_to_move)
    datapolish[i]=datapolish[i].drop(0).reset_index(drop=True)


# Find Significant Features and Cluster Them

In [30]:
def find_sig_correlations(climate_modes,ace_values,mode_data,yearrange): 
    """
    Finds the month for each climate mode that has the strongest correlation with ACE.

    Parameters:
        climate_modes: List of month names (e.g., ['JAN', 'FEB', ..., 'DEC']).
        ace_values: ACE (Accumulated Cyclonic Energy) as a 1D NumPy array.
        mode_data: Dictionary with climate mode names as keys and DataFrames with monthly data as values.

    Returns:
        Significant Features and Months
    """
    monthly_correlations = []
    namess_list = []
    for mode_name in mode_data:
        for month in climate_modes:
            df = mode_data[mode_name].copy()
            if month in df.columns:
                month_series = df[month]
            else:
                continue
            
            # Standardize the data
            standardized_ace = StandardScaler().fit_transform(ace_values.reshape(-1, 1)).flatten()
            standardized_month = StandardScaler().fit_transform(month_series.values.reshape(-1, 1)).flatten()

            corr, pval = pearsonr(standardized_month[:yearrange], standardized_ace[:yearrange])

            combined_name = f"{mode_name}_{month}"
            if pval <= .05:

                monthly_correlations.append(month_series)
                namess_list.append(combined_name)

    data = {name: series for name, series in zip(namess_list, monthly_correlations)}
    dz=pd.DataFrame(data)
    return dz

In [31]:
signearace=find_sig_correlations(MONTH_COLS,nearace_prime,datapolish,70)
sigace=find_sig_correlations(MONTH_COLS,ace_prime,datapolish,70)

In [32]:
def feature_selection_hierarchical_clustering(X, y, threshold):
    # 1. Calculate the correlation matrix (using spearman correlation)
    corr_matrix = X.corr(method='pearson').abs()
    
    # Convert the correlation matrix to a distance matrix
    # Distance = 1 - |correlation|
    distance_matrix = 1 - corr_matrix
    
    # Ensure distance matrix is a condensed distance matrix for scipy linkage
    condensed_distance_matrix = squareform(distance_matrix)
    
    # 2. Perform hierarchical clustering
    linkage_matrix = hierarchy.linkage(condensed_distance_matrix, method='ward') 
    
    # 3. (Optional) Plot dendrogram to determine threshold
    # plt.figure(figsize=(10, 5))
    # hierarchy.dendrogram(linkage_matrix, labels=X.columns.tolist(), leaf_rotation=90)
    # plt.show()
    
    # 4. Cut the dendrogram to form flat clusters based on a threshold
    feature_clusters = hierarchy.fcluster(linkage_matrix, threshold, criterion='distance')
    
    # Map features to their cluster labels
    cluster_df = pd.DataFrame({'feature': X.columns, 'cluster': feature_clusters})
    
    # 5. Select the best feature from each cluster
    selected_features = []
    target_correlations = X.corrwith(y).abs() # Correlation with the target variable

    for cluster_id in sorted(cluster_df['cluster'].unique()):
        # Get features in the current cluster
        features_in_cluster = cluster_df[cluster_df['cluster'] == cluster_id]['feature'].tolist()
        
        # Find the feature with the maximum correlation with the target
        best_feature = target_correlations[features_in_cluster].idxmax()
        selected_features.append(best_feature)
        
    return selected_features

# Example Usage:
# Assuming you have a feature matrix X (pandas DataFrame) and a target variable y (pandas Series)
selected = feature_selection_hierarchical_clustering(sigace[:70], pd.Series(ace_prime[:70]), 0.25)
selected2 = feature_selection_hierarchical_clustering(signearace[:70], pd.Series(nearace_prime[:70]), 0.25)
print(f"Selected ACE features: {selected}")
print(f"Selected CHACE features: {selected2}")

Selected ACE features: ['tsa_JUL', 'tsa_NOV', 'tsa_DEC', 'tsa_MAR', 'mdrolr_NOV', 'roni_MAY', 'whwp_AUG', 'mdrsst_DEC', 'tna_AUG', 'mdrt200_OCT', 'whwp_JUL', 'epo_AUG', 'pna_JUL', 'amo_MAY', 'amo_JUN', 'amo_FEB', 'epo_SEP', 'mdrv200_NOV', 'ngst_SEP', 'sgst_JUN', 'sgst_MAR', 'ngst_APR', 'ngst_FEB', 'nino4_JUN', 'whwp_DEC', 'mdrsst_JAN', 'amm_FEB', 'amm_MAY', 'tna_MAY', 'amm_OCT', 'amm_JAN', 'mdru850_AUG', 'gomslp_SEP', 'mdrvws_JUN', 'mdrslp_OCT', 'gomslp_OCT', 'mdrv200_JUN', 'gomu850_JUL', 'mdrv200_JUL', 'mdrv200_AUG', 'mdrv850_JUL', 'mdrv850_AUG', 'ao_OCT', 'gomolr_MAY', 'nao_MAY', 'mdrslp_JAN', 'pna_DEC', 'ao_APR', 'pna_OCT']
Selected CHACE features: ['amo_APR', 'ngst_OCT', 'ngst_FEB', 'mdrt200_OCT', 'tna_AUG', 'whwp_AUG', 'tna_NOV', 'amm_OCT', 'amo_SEP', 'amo_FEB', 'epo_SEP', 'gomslp_OCT', 'whwp_JUL', 'epo_AUG', 'pna_JUL', 'epo_JUN', 'amm_MAY', 'tna_MAY', 'amm_FEB', 'tna_FEB', 'nino4_JUL', 'soi_JUN', 'mdrolr_OCT', 'whwp_DEC', 'wpi_DEC', 'gomv200_NOV', 'ao_APR', 'mdrv200_JUN', 'gomu85

In [33]:
# Isolate hierarchical values
selected_columns_any = [col for col in sigace.columns if any(s in col for s in selected)]
newsigace = sigace[selected_columns_any]
selected_columns_any = [col for col in signearace.columns if any(s in col for s in selected2)]
newsignearace = signearace[selected_columns_any]

inseason=['JUN','JUL','AUG']
for i in inseason:
    nino34dat=realnino34anom[inseason].iloc[1:].reset_index(drop=True)
    newsigace['nino34_forecast_{}'.format(i)] = nino34dat[i]
    newsignearace['nino34_forecast_{}'.format(i)] = nino34dat[i]




# Save Significant Clustered Features

In [35]:
if ensoremover=='ON':
    newsigace.to_csv(f"{NEW_PATH}noensoclusteredace.csv")
    newsignearace.to_csv(f"{NEW_PATH}noensoclusteredcoastace.csv")
elif ensoremover=='OFF':
    newsigace.to_csv(f"{NEW_PATH}clusteredace.csv")
    newsignearace.to_csv(f"{NEW_PATH}clusteredcoastace.csv")

# For Forecasting with ENSO Removal, Remove ENSO from Incomplete Years

In [193]:
file=xr.open_dataset(f"{BASE_PATH}uwnd.mon.mean.nc")
lon_name='lon'
file['_longitude_adjusted'] = xr.where(file[lon_name] > 180, file[lon_name] - 360, file[lon_name])
file = file.swap_dims({lon_name: '_longitude_adjusted'})
file = file.sel(**{'_longitude_adjusted': sorted(file._longitude_adjusted)})
index1=file.indexes['time'].get_loc('1950-01-01T00:00:00.000000000')
index2=file.indexes['time'].get_loc('2026-02-01T00:00:00.000000000')
file=file.isel(time=slice(index1,index2+1))
uwindfile = file.drop_vars(lon_name)
uwindfile = uwindfile.rename({'_longitude_adjusted': lon_name})


file=xr.open_dataset(f"{BASE_PATH}vwnd.mon.mean.nc")
lon_name='lon'
file['_longitude_adjusted'] = xr.where(file[lon_name] > 180, file[lon_name] - 360, file[lon_name])
file = file.swap_dims({lon_name: '_longitude_adjusted'})
file = file.sel(**{'_longitude_adjusted': sorted(file._longitude_adjusted)})
index1=file.indexes['time'].get_loc('1950-01-01T00:00:00.000000000')
index2=file.indexes['time'].get_loc('2026-02-01T00:00:00.000000000')
file=file.isel(time=slice(index1,index2+1))
vwindfile = file.drop_vars(lon_name)
vwindfile = vwindfile.rename({'_longitude_adjusted': lon_name})

file=xr.open_dataset(f"{BASE_PATH}air.mon.mean.nc")
lon_name='lon'
file['_longitude_adjusted'] = xr.where(file[lon_name] > 180, file[lon_name] - 360, file[lon_name])
file = file.swap_dims({lon_name: '_longitude_adjusted'})
file = file.sel(**{'_longitude_adjusted': sorted(file._longitude_adjusted)})
index1=file.indexes['time'].get_loc('1950-01-01T00:00:00.000000000')
index2=file.indexes['time'].get_loc('2026-02-01T00:00:00.000000000')
file=file.isel(time=slice(index1,index2+1))
airfile = file.drop_vars(lon_name)
airfile = airfile.rename({'_longitude_adjusted': lon_name})


In [195]:
def prepare_forecast_netcdf(ds, var_name, monthstoadd, forecast_year=2026):
    """
    Extends the dataset to the end of the forecast year and 
    pads missing months with the long-term climatology.
    monthstoadd should be two strings of the corresponding 2 digit month (i.e. January = 01)
    """
    # 1. Create a full 12-month time range for the forecast year
    forecast_times = pd.date_range(f"{forecast_year}-{monthstoadd[0]}-01", f"{forecast_year}-{monthstoadd[1]}-01", freq='MS')
    
    # 2. Combine the old time index with the new 2026 index
    full_time_index = ds.time.to_index().union(forecast_times)
    
    # 3. Reindex the dataset (this adds the new months as NaNs)
    ds_extended = ds.reindex(time=full_time_index)
    
    # 4. Padding with Climatology
    # We calculate the mean for every month (Jan-Dec) from the historical record
    climatology = ds_extended.groupby("time.month").mean("time")
    
    # Fill the NaNs in 2026 with the corresponding month's historical mean
    # This keeps the EOF variance stable for the forecast
    ds_padded = ds_extended.groupby("time.month").fillna(climatology)
    
    return ds_padded

# --- Usage ---
uwind_extended = prepare_forecast_netcdf(uwindfile, 'uwnd', ['03', '12'], forecast_year=2026)
vwind_extended = prepare_forecast_netcdf(vwindfile, 'vwnd', ['03', '12'], forecast_year=2026)
air_extended = prepare_forecast_netcdf(airfile, 'air', ['03', '12'], forecast_year=2026)

In [209]:
roni.loc[76]=[2026, -.94, -.7, np.round(roni['MAR'].mean(),2), np.round(roni['APR'].mean(),2),
                  np.round(roni['MAY'].mean(),2),np.round(roni['JUN'].mean(),2),np.round(roni['JUL'].mean(),2),
                  np.round(roni['AUG'].mean(),2),np.round(roni['SEP'].mean(),2),np.round(roni['OCT'].mean(),2),
                   np.round(roni['NOV'].mean(),2),np.round(roni['DEC'].mean(),2)]
u_results200 = process_wind_component(uwind_extended, 'uwnd', roni, areas, 200)
v_results200 = process_wind_component(vwind_extended, 'vwnd', roni, areas, 200)
u_results850 = process_wind_component(uwind_extended, 'uwnd', roni, areas, 850)
v_results850 = process_wind_component(vwind_extended, 'vwnd', roni, areas, 850)
air_results200 = process_wind_component(air_extended, 'air', roni, areas, 200)

# --- Final Wind Shear Calculation ---
# Now calculate VWS using the ENSO-free components
def calc_vws_clean(u200, u850, v200, v850):
    u_diff = u200.iloc[:, 1:].values - u850.iloc[:, 1:].values
    v_diff = v200.iloc[:, 1:].values - v850.iloc[:, 1:].values
    shear = np.sqrt(u_diff**2 + v_diff**2)
    return pd.DataFrame(shear, columns=allmonth)

mdr_vws_clean = calc_vws_clean(u_results200['mdr'], u_results850['mdr'], v_results200['mdr'], v_results850['mdr'])
gom_vws_clean = calc_vws_clean(u_results200['gom'], u_results850['gom'], v_results200['gom'], v_results850['gom'])

# Dump Incomplete Years

In [212]:
allave1 = {}
allave1['mdrt200']=air_results200['mdr']
allave1['mdru200']=u_results200['mdr']
allave1['mdru850']=u_results850['mdr']
allave1['mdrv200']=v_results200['mdr']
allave1['mdrv850']=v_results850['mdr']
allave1['mdrvws']=mdr_vws_clean

allave1['gomu200']=u_results200['gom']
allave1['gomu850']=u_results850['gom']
allave1['gomv200']=v_results200['gom']
allave1['gomv850']=v_results850['gom']
allave1['gomvws']=gom_vws_clean
with open(f"{NEW_PATH}updatednoensowind.pkl", "wb") as f:
    pickle.dump(allave1, f)

# Calculate the CPC Class

In [222]:
def secondcat(thelist,secondlist):
    veryhigh=[]
    high=[]
    mid=[]
    low=[]
    score=[]
    namescore=[]
    # firstquartile = np.percentile(secondlist, 33)
    # thirdquartile = np.percentile(secondlist, 67)
    # highactive = 1.65*np.median(secondlist)
    for i in range(len(thelist)):
        if (thelist[i] < 73):
            low.append(yearzz[i])
            score.append(0)
            namescore.append('Below Average')
        elif (thelist[i] >= 126.1) and (thelist[i] <= 159.6):
            high.append(yearzz[i])
            score.append(2)
            namescore.append('Above Average')
        elif (thelist[i] > 159.6):
            veryhigh.append(yearzz[i])
            score.append(3)
            namescore.append('Extremely Active')
        else:
            mid.append(yearzz[i])
            score.append(1)
            namescore.append('Near Average')
    return veryhigh, high, mid, low, score, namescore

In [223]:
aceveryhigh,acehigh,acemid,acelow,acescore,acenamescore=secondcat(ace_prime,ace_prime[:70])

In [224]:
cpcdump=[pd.Series(aceveryhigh),pd.Series(acehigh),pd.Series(acemid),pd.Series(acelow),pd.Series(acenamescore),pd.Series(acescore)]

In [ ]:
with open(f"{NEW_PATH}cpcclass.pkl", "wb") as f:
    pickle.dump(cpcdump, f)